In [11]:
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

RANDOM_STATE = 42
MULTIPLIER_100 = 100

# fetch dataset
adult = fetch_ucirepo(id=2)

# data (as pandas dataframes)
X = adult.data.features.copy() # copy to avoid side effects.
y = adult.data.targets.squeeze().copy()  # convert to 1D array,
                                         # copy to avoid side effects.

###############################
# Dataset cleanup
###############################

# normalize target labels, to only have two target classes: <=50K and >50K
y = y.str.strip().str.rstrip(".")
# replace missing "?" values with NaN
X = X.replace("?", np.nan)

###############################
# Inspect the dataset
###############################

print("\nDimensionality of the dataset: ")
print(X.shape)

print("\nDataset info: ")
X.info()

num_summary = X.describe().T
print("\nNumerical features: ")
print(num_summary)

categ_summary = X.describe(include=["str", "category"]).T
print("\nCategorical features: ")
print(categ_summary)

print("\nTarget values: ")
print(y.value_counts())

print("\nTarget proportions (%): ")
print(y.value_counts(normalize=True) * MULTIPLIER_100)

print("\nMissing values: ")
print(X.isna().sum())

print("\nTotal missing values in X:")
print(X.isna().sum().sum())

print("\nMissing values in y:")
print(y.isna().sum())

missing_summary = pd.DataFrame({
    "Missing count": X.isna().sum(),
    "Missing (%)": X.isna().mean() * MULTIPLIER_100
})

missing_summary = missing_summary[missing_summary["Missing count"] > 0]
print(missing_summary)

print("\nOccupation missingness in workclass:")
occupation_missing_in_workclass = pd.crosstab(
    X["workclass"].fillna("Missing"),
    X["occupation"].isna(),
    normalize="index"
) * MULTIPLIER_100
print(occupation_missing_in_workclass)

print("\nNative country missingness in workclass:")
native_contry_missing_in_workclass = pd.crosstab(
    X["workclass"].fillna("Missing"),
    X["native-country"].isna(),
    normalize="index"
) * MULTIPLIER_100
print(native_contry_missing_in_workclass)

print("\nJoint missingness of workclass and occupation:")
joint_missing = pd.crosstab(
    X["workclass"].isna(),
    X["occupation"].isna(),
)
print(joint_missing)

###############################
# 60/20/20 stratified split
###############################

# training set, using 60% of the data
X_train, X_temp, y_train, y_temp = train_test_split(X,
                                                    y,
                                                    test_size=0.4,
                                                    stratify=y, # stratify using target label
                                                                # that corresponds to the data
                                                                # that is split.
                                                    random_state=RANDOM_STATE)

# validation set and test set, each using 20% of the data
X_val, X_test, y_val, y_test = train_test_split(X_temp,
                                                y_temp,
                                                test_size=0.5,
                                                stratify=y_temp, # stratify only on the temporary 40%
                                                                 # of the data.
                                                random_state=RANDOM_STATE)

###############################
# Verify splits
###############################

print("\nTraining set size: ", X_train.shape)
print("\nValidation set size: ", X_val.shape)
print("\nTest set size: ", X_test.shape)

print("\nTraining set target proportions (%): ")
print(y_train.value_counts(normalize=True) * MULTIPLIER_100)

print("\nValidation set target proportions (%): ")
print(y_val.value_counts(normalize=True) * MULTIPLIER_100)

print("\nTest set target proportions (%): ")
print(y_test.value_counts(normalize=True) * MULTIPLIER_100)

###############################
# Identify feature types
###############################

num_features = X_train.select_dtypes(
    include=["number"]
).columns

categ_features = X_train.select_dtypes(
    include=["str", "category"]
).columns

print("\nNumerical features: ")
print(num_features.tolist())

print("\nCategorical features: ")
print(categ_features.tolist())

######################################
# 5-fold stratified cross-validation
######################################

skfolds = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

###############################
# Preprocessing pipelines
###############################

# Support Vector Machine (SVM) pipeline.
# Fill missing numerical values with median and standardize numerical features
num_pipeline_svm = make_pipeline(
    SimpleImputer(strategy="median"), # fill missing numerical values with median
    StandardScaler() # apply standardization to numerical features
)

# Decision Tree (DT) pipeline.
# Fill missing numerical values only
num_pipeline_tree = make_pipeline(
    SimpleImputer(strategy="median") # fill missing numerical values with median
)

# fill missing categorical values and one-hot encode categorical features
cat_pipeline = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="Missing"),
    OneHotEncoder(handle_unknown="ignore")
)

###############################################
# Preprocessing for SVM and DT
###############################################

svm_preprocessing = ColumnTransformer([
        ("num", num_pipeline_svm, num_features),
        ("cat", cat_pipeline, categ_features)
])

tree_preprocessing = ColumnTransformer([
    ("num", num_pipeline_tree, num_features),
    ("cat", cat_pipeline, categ_features)
])


Dimensionality of the dataset: 
(48842, 14)

Dataset info: 
<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             48842 non-null  int64
 1   workclass       46043 non-null  str  
 2   fnlwgt          48842 non-null  int64
 3   education       48842 non-null  str  
 4   education-num   48842 non-null  int64
 5   marital-status  48842 non-null  str  
 6   occupation      46033 non-null  str  
 7   relationship    48842 non-null  str  
 8   race            48842 non-null  str  
 9   sex             48842 non-null  str  
 10  capital-gain    48842 non-null  int64
 11  capital-loss    48842 non-null  int64
 12  hours-per-week  48842 non-null  int64
 13  native-country  47985 non-null  str  
dtypes: int64(6), str(8)
memory usage: 5.2 MB

Numerical features: 
                  count           mean            std      min       25%  \
age      